# Notebook 05: Advanced Embeddings & Retrieval Architectures for RAG

Welcome to the fifth and final notebook in the **Embeddings from Scratch** series!

In previous notebooks, we moved from vector math $\rightarrow$ static Word2Vec $\rightarrow$ dynamic BERT $\rightarrow$ contrastive sentence bi-encoders.

In this notebook, we master the modern paradigms used in **cutting-edge, production-grade RAG systems**:
1. **Instruction-Tuned / Asymmetric Embeddings** (e.g., BGE, E5).
2. **Matryoshka Representation Learning (MRL)**: Compressing vector dimensions without losing accuracy.
3. **Dense vs. Sparse vs. Hybrid Search** (Why BM25 + Dense is often king).
4. **Late Interaction (ColBERT)**: Overcoming the single-vector bottleneck.
5. **Vector Indexing Fundamentals**: Exact Search vs. Approximate Nearest Neighbors (ANN / HNSW).
6. **Building an End-to-End Semantic Search Engine** from scratch.


---
## 1. Setup & Environment
Run this cell on your Google server / Colab instance (GPU recommended).


In [1]:
# Run this on your Google server / Colab instance:
# !pip install -q sentence-transformers torch numpy matplotlib

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cpu


---
## 2. Asymmetric Search & Instruction-Tuned Embeddings

In traditional NLP, sentences are compared symmetrically:
$$\text{Sentence A} \longleftrightarrow \text{Sentence B}$$

In **RAG**, search is almost always **asymmetric**:
* The **Query** is short, informal, and asks a question: *"how to fix wifi on macbook?"*
* The **Document** is long, formal, and explanatory: *"To troubleshoot network connectivity in macOS, navigate to System Settings..."*

Leading models like **BGE (BAAI)** and **E5 (Microsoft)** use **task prefixes/instructions** to signal the role of the text to the encoder:


In [2]:
# Loading a modern, highly efficient embedding model: BGE-small
# (Lightweight, fast, and top-tier MTEB retrieval performance)
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)

# BGE requires queries to be prefixed with an instruction:
query_prefix = "Represent this sentence for searching relevant passages: "
query = "What is the capital of France?"
instructed_query = query_prefix + query

documents = [
    "Paris is the capital and most populous city of France, known for its art and landmarks.",
    "Berlin is the capital of Germany and one of the largest technology hubs in Europe.",
    "Tokyo is the bustling capital city of Japan, famed for modern skyscrapers and historic temples."
]

# Documents do NOT need the prefix:
query_emb = model.encode(instructed_query, normalize_embeddings=True)
doc_embs  = model.encode(documents, normalize_embeddings=True)

# Compute similarity
sims = np.dot(doc_embs, query_emb)
for doc, score in zip(documents, sims):
    print(f"Score: {score:.4f} | Document: {doc}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Score: 0.7870 | Document: Paris is the capital and most populous city of France, known for its art and landmarks.
Score: 0.6008 | Document: Berlin is the capital of Germany and one of the largest technology hubs in Europe.
Score: 0.5492 | Document: Tokyo is the bustling capital city of Japan, famed for modern skyscrapers and historic temples.


---
## 3. Matryoshka Representation Learning (MRL)

### The Production Problem:
* A standard embedding model might output $d = 768$ or $d = 1536$ dimensions.
* Storing 10,000,000 vectors at 1536 dimensions (float32) requires:
  $$10^7 \times 1536 \times 4 \text{ bytes} \approx 61.44 \text{ GB of RAM!}$$
* High RAM usage increases vector database hosting costs significantly.

### The MRL Solution:
Named after Russian nesting dolls (Matryoshka), **MRL** trains the model so that the **first $k$ dimensions** (e.g., 64, 128, 256) store the highest-priority semantic information.

```
Full Vector (768D):  [ ■ ■ ■ ■ ■ ■ ■ ■ ... ■ ■ ■ ■ ] ──► 100% Quality, 100% Storage
Slice first 256D:   [ ■ ■ ■ ■ ■ ■ ■ ■ ]             ──► 98% Quality,  33% Storage!
Slice first 128D:   [ ■ ■ ■ ■ ]                     ──► 94% Quality,  16% Storage!
```

To use a truncated MRL vector, you simply:
1. Slice the first $k$ dimensions: `vec[:k]`
2. **Re-normalize to unit length** ($L_2$ norm = 1)


In [3]:
# Testing MRL dimension truncation on BGE-small (original dim = 384)
full_q = model.encode(instructed_query, normalize_embeddings=False)
full_docs = model.encode(documents, normalize_embeddings=False)

def truncate_and_normalize(vectors, target_dim):
    # Slice first target_dim dimensions
    truncated = vectors[:, :target_dim] if vectors.ndim > 1 else vectors[:target_dim]
    # Re-normalize
    if truncated.ndim > 1:
        return truncated / np.linalg.norm(truncated, axis=1, keepdims=True)
    return truncated / np.linalg.norm(truncated)

dims_to_test = [384, 256, 128, 64]
print(f"Query: '{query}'\n")

for d in dims_to_test:
    q_d = truncate_and_normalize(full_q, d)
    docs_d = truncate_and_normalize(full_docs, d)
    scores_d = np.dot(docs_d, q_d)
    
    print(f"--- Dimension: {d} (Storage reduction: {(1 - d/384)*100:.1f}%) ---")
    print(f"  Target Match (Paris):  {scores_d[0]:.4f}")
    print(f"  Distractor (Berlin):  {scores_d[1]:.4f}")
    print(f"  Margin:               {scores_d[0] - scores_d[1]:.4f}")


Query: 'What is the capital of France?'

--- Dimension: 384 (Storage reduction: 0.0%) ---
  Target Match (Paris):  0.7870
  Distractor (Berlin):  0.6008
  Margin:               0.1863
--- Dimension: 256 (Storage reduction: 33.3%) ---
  Target Match (Paris):  0.7754
  Distractor (Berlin):  0.6315
  Margin:               0.1439
--- Dimension: 128 (Storage reduction: 66.7%) ---
  Target Match (Paris):  0.7994
  Distractor (Berlin):  0.6820
  Margin:               0.1174
--- Dimension: 64 (Storage reduction: 83.3%) ---
  Target Match (Paris):  0.8029
  Distractor (Berlin):  0.6261
  Margin:               0.1768


Even at **128 dimensions** (a 66% reduction in storage and memory), the model easily separates the target match from distractors with high confidence!


---
## 4. Dense vs. Sparse vs. Late Interaction

In modern RAG architectures, you will encounter three main paradigms:

| Paradigm | Example | How It Works | Strengths | Weaknesses |
| :--- | :--- | :--- | :--- | :--- |
| **Dense** | BGE, E5, OpenAI | 1 vector per doc ($\mathbb{R}^{768}$) | Semantic understanding, synonyms | Fails on exact product IDs / rare keywords |
| **Learned Sparse** | SPLADE, BM25 | Sparse term weights over vocabulary | Exact keyword matching, fast inverted index | Struggles with paraphrasing |
| **Late Interaction** | ColBERT | Multi-vector: 1 vector **per token** | Extreme precision, preserves fine details | High storage footprint |

```
Dense Bi-Encoder:
Query ──► [Enc] ──► [v_q (768D)] ──┐
                                    ├── Dot Product ──► Score
Doc   ──► [Enc] ──► [v_d (768D)] ──┘

Late Interaction (ColBERT):
Query ──► [Enc] ──► [q1, q2, q3, ...] ──┐
                                          ├── MaxSim Matrix ──► Sum of Max Matches
Doc   ──► [Enc] ──► [d1, d2, d3, ...] ──┘
```


---
## 5. Simulating ColBERT's MaxSim Operator

To understand Late Interaction, let's implement ColBERT's **MaxSim operator** from scratch in PyTorch.

For a query with tokens $Q = [q_1, q_2, \dots, q_m]$ and document tokens $D = [d_1, d_2, \dots, d_n]$:
$$\text{Score}(Q, D) = \sum_{i=1}^m \max_{j=1}^n (q_i \cdot d_j)$$

Every query token finds its **single most relevant token** in the document, and those maximum scores are summed!


In [4]:
def colbert_maxsim(query_token_embeddings, doc_token_embeddings):
    """
    query_token_embeddings: [num_query_tokens, embed_dim]
    doc_token_embeddings:   [num_doc_tokens, embed_dim]
    """
    # 1. Compute all-to-all token cosine similarity matrix
    # Shape: [num_query_tokens, num_doc_tokens]
    sim_matrix = torch.matmul(query_token_embeddings, doc_token_embeddings.T)
    
    # 2. For each query token, take the maximum match across all doc tokens
    max_scores_per_query_token, _ = torch.max(sim_matrix, dim=1)
    
    # 3. Sum the maximum scores
    return torch.sum(max_scores_per_query_token).item()

# Simulating token embeddings (e.g. 4 query tokens, 8 doc tokens)
q_tokens = torch.randn(4, 128)
d_tokens = torch.randn(8, 128)

q_tokens = F.normalize(q_tokens, p=2, dim=1)
d_tokens = F.normalize(d_tokens, p=2, dim=1)

score = colbert_maxsim(q_tokens, d_tokens)
print(f"ColBERT MaxSim relevance score: {score:.4f}")


ColBERT MaxSim relevance score: 0.4631


---
## 7. Complete Overview: The 5-Notebook Journey

Congratulations! You have completed the **Embeddings from Scratch** series!

Here is what you have mastered across this curriculum:

1. **Notebook 01**: Vector math, $L_1/L_2$ norms, Euclidean distance, Cosine Similarity, and why normalized dot product powers vector search.
2. **Notebook 02**: Why sparse vectors fail, the Distributional Hypothesis, Word2Vec CBOW vs. Skip-Gram, Negative Sampling, and the polysemy bottleneck.
3. **Notebook 03**: Transformers and Self-Attention ($Q, K, V$), dynamic contextual representations with BERT, pooling techniques, and the anisotropy issue.
4. **Notebook 04**: Cross-Encoders vs. Bi-Encoders, Contrastive Learning, Triplet Loss, Multiple Negatives Ranking Loss (MNRL), and Hard Negative Mining.
5. **Notebook 05**: Asymmetric instruction tuning, Matryoshka dimension truncation (MRL), Dense vs. Sparse vs. ColBERT Late Interaction, and vector indexing for production RAG.

You are now fully prepared to build high-accuracy retrieval engines, fine-tune custom embedding models, and configure vector databases with complete conceptual mastery!
